# Lab 10 · LangGraph basics: state, nodes and edges, with the small model from the first line

**Time:** 40 min · **Work in teams of 3** · **You will:** build a small **LangGraph** workflow in which **every node that thinks calls the small model**. You will define the **state**, write **nodes**, connect them with a **normal edge** and a **conditional edge**, run it with **`invoke`** and **`stream`**, draw it, and compare it with the `create_agent` graph from Lab 9.

**Why it matters:** in Lab 9 the model decided every step. Here **you** decide the shape of the work and the model does the language part inside the nodes. That is what makes a workflow predictable and testable.

**The idea of the lab:** *a graph is a state, some nodes, and edges that say what runs next.* That is all of LangGraph, and the next four labs only add to it.

*How to use this notebook: **Kernel → Restart Kernel and Run All Cells**. Predict before each run.*

In [ ]:
import sys, os
sys.path.insert(0, next(os.path.abspath(d) for d in ("..", "../..") if os.path.exists(os.path.join(d, "slm_lab.py"))))
from slm_lab import check
from typing import TypedDict
from lc_slm import SmallChatModel
from graph_viz import draw_graph
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import SystemMessage, HumanMessage

router_model = SmallChatModel(max_new_tokens=8)        # the small model for one-word answers
talk_model = SmallChatModel(max_new_tokens=80)         # the same small model, for short replies
print("Both are the same Qwen2.5-3B on your GPU:", router_model.invoke("Reply with the word ready.").content)

## 10.1 Your first graph: one node that calls the small model
A graph passes a **state** from node to node. The state is a typed dictionary: every node reads it and **returns only the keys it changes**.

The first node, `classify`, asks the small model one question: *what does the user want: `act` (do something in the home), `ask` (a question) or `chat` (small talk)?*

In [ ]:
ROUTES = ["act", "ask", "chat"]

class State(TypedDict, total=False):
    message: str          # what the user said
    route: str            # the model's label
    reply: str            # what we will say back

CLASSIFY_PROMPT = ("You route messages for a smart-home assistant. Reply with ONE word.\n"
                   "act = the user wants something done in the home (lights, AC, blinds)\n"
                   "ask = the user asks a question about the home or a device\n"
                   "chat = greeting, thanks or small talk")

def classify(state: State) -> dict:
    """Node 1: the small model labels the message."""
    out = router_model.invoke([SystemMessage(CLASSIFY_PROMPT), HumanMessage(state["message"])]).content.strip().lower()
    label = next((r for r in ROUTES if out.startswith(r)), "chat")       # the code checks the label
    return {"route": label}

first = StateGraph(State)
first.add_node("classify", classify)
first.add_edge(START, "classify")           # a normal edge: always go here
first.add_edge("classify", END)
first_app = first.compile()

for msg in ["Turn on the kitchen light", "What does error CH05 mean?", "Thanks a lot!"]:
    print(f"{msg!r:35} ->", first_app.invoke({"message": msg}))

**Read it:** the state that comes back holds the input (`message`) **and** what the node added (`route`). The node returned only `{"route": ...}`: LangGraph merged it into the state. Note the line with `next(...)`: the model's word is **checked in code** and anything unexpected becomes `chat`.

## 10.2 Add the nodes it can reach, each with the small model
Three more nodes, one per route. Every one of them calls the model, with a different job:
- `act` turns the request into a short **plan** (what, where, value);
- `answer` answers a question in one or two sentences;
- `chat` replies in a friendly way.

In [ ]:
def act(state: State) -> dict:
    plan = talk_model.invoke([SystemMessage("You are a smart-home assistant. In one short line, say exactly what you will do: device, room, value. Do not add anything else."),
                              HumanMessage(state["message"])]).content.strip()
    return {"reply": "PLAN: " + plan}

def answer(state: State) -> dict:
    text = talk_model.invoke([SystemMessage("You are a smart-home assistant. Answer in one or two sentences. If you do not know, say so."),
                              HumanMessage(state["message"])]).content.strip()
    return {"reply": text}

def chat(state: State) -> dict:
    text = talk_model.invoke([SystemMessage("You are a friendly smart-home assistant. Reply in one short sentence."),
                              HumanMessage(state["message"])]).content.strip()
    return {"reply": text}

## 10.3 A conditional edge: the graph chooses the next node
A **conditional edge** is a function that **reads the state and returns the name of the next node**. This is where a workflow differs from a ReAct loop: *you* wrote the choices.

In [ ]:
def pick(state: State) -> str:
    """The routing function: read the label, return the name of the next node."""
    return state["route"]

g = StateGraph(State)
g.add_node("classify", classify)
g.add_node("act", act)
g.add_node("answer", answer)
g.add_node("chat", chat)
g.add_edge(START, "classify")
g.add_conditional_edges("classify", pick, {"act": "act", "ask": "answer", "chat": "chat"})   # label -> node name
g.add_edge("act", END)
g.add_edge("answer", END)
g.add_edge("chat", END)
app = g.compile()

for msg in ["Make the bedroom 22 degrees", "How many degrees can the AC go down to?", "Good morning!"]:
    out = app.invoke({"message": msg})
    print(f"{msg!r:45} route={out['route']:5} reply={out['reply'][:70]!r}")

## 10.4 Watch it run: `invoke` and `stream`
`invoke` gives only the final state. `stream` gives **one update per node, in order**: the best way to see what a graph does.

In [ ]:
for update in app.stream({"message": "Turn off the study light"}, stream_mode="updates"):
    for node, change in update.items():
        print(f"node {node!r:10} returned {change}")

In [ ]:
check(set(next(iter(u)) for u in app.stream({"message": "Turn off the study light"}, stream_mode="updates")) >= {"classify"},
      "stream showed the classify node and then the node the graph chose. Two updates, one per node that ran.",
      "stream should print at least the classify node.")

## 10.5 Draw it, and draw the agent from Lab 9
A compiled graph can draw itself. **Solid arrows** are normal edges, **dashed arrows** are conditional edges. Then look at the `create_agent` graph from Lab 9: it is built from the same pieces.

In [ ]:
draw_graph(app, "Your graph: classify, then one of three nodes")

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent

@tool
def set_light(room: str, state: str) -> str:
    """Turn the light on or off in a room. state is 'on' or 'off'."""
    return f"The {room} light is {state}."

agent = create_agent(model=SmallChatModel(), tools=[set_light], system_prompt="You control a smart home.")
draw_graph(agent, "The create_agent graph from Lab 9")

**Read the two pictures:**
- Yours has a **conditional edge** after `classify` and three ways to the end.
- `create_agent` has **two nodes**, `model` and `tools`, and a **loop**: after `tools` it goes back to `model`. **That loop is ReAct.**

So `create_agent` is a LangGraph graph. LangGraph is the toolkit; ReAct is one graph you can draw with it.

## 10.6 Measure the router
A router is a model call, so it needs a score. Twelve labelled messages, including four that need a little inference (*it is getting warm in here* does not name a device):

In [ ]:
LABELLED = [
    ("Turn on the kitchen light", "act"), ("Set the bedroom AC to 21", "act"), ("Close the blinds in the study", "act"),
    ("It is getting warm in here", "act"), ("The study is too bright", "act"),
    ("What does error CH05 mean?", "ask"), ("How do I reset the AC filter alarm?", "ask"), ("Is the balcony light on?", "ask"),
    ("What is the lowest temperature the AC can do?", "ask"),
    ("Hello there", "chat"), ("Thanks, that was helpful", "chat"), ("I am home!", "chat"),
]
right = 0
for msg, want in LABELLED:
    got = classify({"message": msg})["route"]
    right += got == want
    if got != want:
        print(f"WRONG  {msg!r:50} wanted {want}, model said {got}")
router_score = right / len(LABELLED)
print(f"\nrouter accuracy: {right}/{len(LABELLED)} = {router_score:.0%}")
check(router_score >= 0.6,
      f"The small model routes {router_score:.0%} of the messages correctly. The graph does the rest of the work with that label, and the code can still override it (next task).",
      "the router got fewer than 7 of 12: read the WRONG lines to see which kind of message it confuses.")

## 10.7 Your turn: let the code override the model
### 🔧 TODO 10 — Route dangerous requests to `refuse`
A model's label should never be the **only** thing between a request and an action. Add a fourth route, `refuse`, chosen **by your code** when the message mentions the **door** or a **lock**, whatever the model said.

In [ ]:
def refuse(state: State) -> dict:
    return {"reply": "I cannot do that. Please ask the owner."}

def pick2(state: State) -> str:
    # TODO: if the message mentions "door" or "lock" (any capitals), return "refuse"
    # otherwise return the model's label, as pick did
    return state["route"]                       # this line already works for normal messages; add your rule above it

g2 = StateGraph(State)
g2.add_node("classify", classify); g2.add_node("act", act); g2.add_node("answer", answer)
g2.add_node("chat", chat); g2.add_node("refuse", refuse)
g2.add_edge(START, "classify")
g2.add_conditional_edges("classify", pick2, {"act": "act", "ask": "answer", "chat": "chat", "refuse": "refuse"})
for n in ("act", "answer", "chat", "refuse"):
    g2.add_edge(n, END)
app2 = g2.compile()
print(app2.invoke({"message": "Unlock the front door for the delivery man"}))

In [ ]:
out_door = app2.invoke({"message": "Unlock the front door for the delivery man"})
out_light = app2.invoke({"message": "Turn on the kitchen light"})
check(out_door["reply"].startswith("I cannot") and not out_light["reply"].startswith("I cannot"),
      "The door request was refused by your code, and a normal request still went through. The model proposes a label; your function decides.",
      "the door request should end in the refuse node and the kitchen light request should not.")

## 10.8 What you proved
- **State** carries the data; each node returns only what it changes.
- **Normal edges** say "always next". A **conditional edge** reads the state and chooses.
- Every node that needs language called the **small model**; your code checked the label and could override it.
- `create_agent` is a graph of the same pieces, with a loop. **ReAct is one graph; LangGraph lets you draw your own.**

## Check your understanding
1. What does a node return, and what does LangGraph do with it?
2. What is the difference between `add_edge` and `add_conditional_edges`? Give one use of each from this lab.
3. In the TODO the code overrode the model's label. Why is that a good habit?
4. Looking at the two pictures: where is the loop in `create_agent`, and what does it correspond to in Lab 9?

*✍️ Your answer:* 

## Next
Lab 11 adds the rest of the vocabulary: a **retry loop**, **reducers**, **Command**, the **recursion limit** and **fan-out** (parallel branches).